In [3]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

repair_types = {
    "Oil Change": 2,
    "Brake Repair": 6,
    "Tyre Replacement": 3,
    "Engine Repair": 36,
    "Transmission Repair": 48,
    "Electrical Repair": 8,
    "Suspension Repair": 12,
    "Battery Replacement": 2,
    "Clutch Repair": 18
}

vehicle_types = ["Truck", "Van", "Bus", "Mini Truck"]

rows = []

for i in range(5000):

    vehicle_type = random.choice(vehicle_types)

    age = random.randint(1, 12)

    odometer = random.randint(age*25000, age*50000)

    maintenance_count = random.randint(0, age+8)

    repair_type = random.choice(list(repair_types.keys()))

    severity = random.choice(
        ["Low","Medium","High"]
    )

    parts_available = random.choice([0,1])

    technician_experience = random.randint(1,15)

    workshop_load = random.randint(1,10)

    base = repair_types[repair_type]

    repair_time = base

    if severity=="Low":
        repair_time += 0

    elif severity=="Medium":
        repair_time += 6

    else:
        repair_time += 18

    repair_time += age * 0.8

    repair_time += odometer/100000

    repair_time += maintenance_count*0.5

    if parts_available==0:
        repair_time += random.randint(12,30)

    repair_time += workshop_load*1.3

    repair_time -= technician_experience*0.4

    repair_time += np.random.normal(0,2)

    repair_time=max(1,round(repair_time,2))

    rows.append({

        "Vehicle_Type":vehicle_type,
        "Vehicle_Age":age,
        "Odometer":odometer,
        "Maintenance_Count":maintenance_count,
        "Repair_Type":repair_type,
        "Severity":severity,
        "Parts_Available":parts_available,
        "Technician_Experience":technician_experience,
        "Workshop_Load":workshop_load,
        "Repair_Time_Hours":repair_time

    })

df=pd.DataFrame(rows)

df.to_csv("maintenance_dataset.csv",index=False)

print(df.head())
print(df.shape)

  Vehicle_Type  Vehicle_Age  Odometer  Maintenance_Count          Repair_Type  \
0        Truck            1     49299                  4        Engine Repair   
1        Truck           10    360604                  1           Oil Change   
2          Van           12    585706                 13        Engine Repair   
3          Van           12    521571                 10  Transmission Repair   
4        Truck            7    200353                 11    Electrical Repair   

  Severity  Parts_Available  Technician_Experience  Workshop_Load  \
0      Low                0                     12              2   
1      Low                0                      4              9   
2   Medium                1                     13              1   
3      Low                0                     13              6   
4     High                1                     13              1   

   Repair_Time_Hours  
0              67.09  
1              35.93  
2              61.35  
3     

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error,r2_score


df=pd.read_csv("maintenance_dataset.csv")

X=df.drop("Repair_Time_Hours",axis=1)

y=df["Repair_Time_Hours"]


categorical=[
"Vehicle_Type",
"Repair_Type",
"Severity"
]

numeric=[
"Vehicle_Age",
"Odometer",
"Maintenance_Count",
"Parts_Available",
"Technician_Experience",
"Workshop_Load"
]


preprocessor=ColumnTransformer(

[
("cat",OneHotEncoder(handle_unknown="ignore"),categorical),
("num","passthrough",numeric)
]

)


model=RandomForestRegressor(

n_estimators=300,
random_state=42

)


pipeline=Pipeline(

[
("preprocessor",preprocessor),
("model",model)
]

)

X_train,X_test,y_train,y_test=train_test_split(

X,
y,
test_size=0.2,
random_state=42

)

pipeline.fit(X_train,y_train)

pred=pipeline.predict(X_test)

print("MAE:",mean_absolute_error(y_test,pred))

print("R2 Score:",r2_score(y_test,pred))


MAE: 4.018798566666668
R2 Score: 0.938172915218844
